라이브러리 설치

In [ ]:
!pip install -q transformers datasets scikit-learn pandas accelerate

2.csv파일 업로드

In [ ]:
from google.colab import files

uploaded = files.upload()
'''
업로드할 csv 이름
github_vulnerability_dataset.csv
csv 구조
code,vuln_type
"os.system('ping ' + user_input)",COMMAND_INJECTION
"query = 'SELECT * FROM users WHERE id=' + user_input",SQL_INJECTION
"element.innerHTML = user_input",XSS
"cursor.execute('SELECT * FROM users WHERE id=?', (user_input,))",SAFE
'''

KeyboardInterrupt: 

3.기본 설정

In [ ]:
import pandas as pd
import numpy as np
import torch
import json

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

DATA_PATH = "github_vulnerability_dataset.csv"
MODEL_NAME = "microsoft/codebert-base"
OUTPUT_DIR = "./codebert_vuln_type_model"

MAX_LENGTH = 256

print("설정 완료")

4.데이터 불러오기 및 라벨 변환

In [ ]:
#csv 파일 불러옴
df = pd.read_csv(DATA_PATH)

print("전체 데이터 수:", len(df))
print("컬럼 목록:", df.columns.tolist())

if "code" not in df.columns or "vuln_type" not in df.columns:
    raise ValueError("CSV 파일에는 반드시 'code', 'vuln_type' 컬럼이 있어야 합니다.")
#결측치 제거
df = df.dropna(subset=["code", "vuln_type"])

#문자열 변환
df["code"] = df["code"].astype(str)
df["vuln_type"] = df["vuln_type"].astype(str)

#취약점 종류별 데이터 개수 확인
print("\n취약점 유형 분포")
print(df["vuln_type"].value_counts())

#취약점 종류 목록 추출
label_list = sorted(df["vuln_type"].unique())

#문자열->숫자 변환
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

df["label"] = df["vuln_type"].map(label2id)

NUM_LABELS = len(label_list)

print("\n라벨 매핑")
print(label2id)

display(df.head())

5. 학습/검증 데이터 분리 및 토큰화

In [ ]:
#전체 데이터 학습용(train),검증용(validation)으로 나누기
train_df, valid_df = train_test_split(
    df,
    test_size=0.2, #데이터 20퍼를 검증용으로 사용
    random_state=42,
    stratify=df["label"]  #특정 취약점만 너무 많아지는 것 방지
)

print("학습 데이터 수:", len(train_df))
print("검증 데이터 수:", len(valid_df))

# pandas dataframe huggingface dataset형태로 변환
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

# codebert 전용 tokenizer 불러오기
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#코드 AI입력 형태로 변환하는 함수
def tokenize_function(batch):
    return tokenizer(
        batch["code"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

#모든 코드 데이터에 tokenizer 적용
train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)

#학습에 필요 없는 컬럼 제거
remove_cols = [
    col for col in train_dataset.column_names
    if col not in ["input_ids", "attention_mask", "label"]
]

#Pytorch Tensor 형태로 변환 codeBERT는 Pytorch기반 모델이라 Tensor 데이터 형태여야지 학습이 가능
train_dataset = train_dataset.remove_columns(remove_cols)
valid_dataset = valid_dataset.remove_columns(remove_cols)

train_dataset.set_format("torch")
valid_dataset.set_format("torch")

print("토큰화 완료")

6.CodeBERT 모델 생성 및 학습

In [ ]:
#CodeBERT 기본 지식 +취약점 데이터셋= 취약점 분류 모델 :기존 모델 가져와서 취약점 유형 분류 모델로 바꾸는 부분
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

#모델이 얼마나 잘 맞 췄는지 평가하는 함수
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

'''accuracy:전체 중 맞춘 비율
precision:취약하다고 예측 한 것중 실제로 맞은 비율
recall:실제 취약 코드 중 모델이 찾아낸 비율
f1:precision과 recall의 균형 점수
'''

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

#모델 어떻게 학습시킬 지 정하는 설정 값
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=8, #한 번에 학습할 데이터 개수, GPU 메모리가 부족하면 4로 줄이면 됨
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,

    logging_steps=10,  #10 step마다 학습 로그 출력
    logging_dir="./logs",

    load_best_model_at_end=True,
    metric_for_best_model="f1", #가장 좋은 모델 고르는 기준, f1점수는 높을수록 좋은 지표
    greater_is_better=True,

    report_to="none" #외부 로깅 도구 사용 안 함
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(f"{OUTPUT_DIR}/label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "label2id": label2id,
            "id2label": id2label
        },
        f,
        ensure_ascii=False,
        indent=4
    )

print("모델 학습 및 저장 완료")
print("저장 위치:", OUTPUT_DIR)

7. 학습된 모델로 예측 테스트

In [ ]:
MODEL_PATH = "./codebert_vuln_type_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# 저장된 CodeBERT 취약점 분류 모델 불러오기
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

with open(f"{MODEL_PATH}/label_mapping.json", "r", encoding="utf-8") as f:
    label_map = json.load(f)

id2label = {int(k): v for k, v in label_map["id2label"].items()}

def predict_vulnerability_type(code):
  #GPU사용 가능하면 GPU사용,안되면 CPU 사용
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)
    model.eval()
#입력 코드 토큰화(새로 넣은 데이터 한개만 토큰화)
    inputs = tokenizer(
        code,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=256
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
#예측할 때는 기울기 계산x
    with torch.no_grad():
        outputs = model(**inputs)
        #모델 원시 출력 값인 logits를 확률로 변환
        probs = torch.softmax(outputs.logits, dim=1)
    #가장 높은 확률 가진 취약점 선택
    pred_id = torch.argmax(probs, dim=1).item()
    pred_label = id2label[pred_id]
    confidence = probs[0][pred_id].item()

    return {
        "prediction": pred_label,
        "confidence": round(confidence, 4),
        "is_vulnerable": pred_label != "SAFE"
    }

test_code = """
import os

user_input = input("IP 입력: ")
os.system("ping " + user_input)
"""

result = predict_vulnerability_type(test_code)
print(result)